In [1]:
import pandas as pd
import cv2 as cv
import rasterio
import os
import numpy as np
import warnings

# Suppress runtime warnings for divisions by zero during masking
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide")
warnings.filterwarnings("ignore", message="invalid value encountered in divide")

# --- CORE ANALYSIS LOGIC ---

def get_segmented_stats(img_name, r, g, b, nir=None, is_satellite=True):
    r, g, b = r.astype(float), g.astype(float), b.astype(float)
    
    if is_satellite and nir is not None:
        nir = nir.astype(float)
        index = (nir - r) / (nir + r + 1e-5) 
        threshold = 0.2 
    else:
        index = (2.0 * g - r - b) / (2.0 * g + r + b + 1e-5) 
        threshold = 0.05 

    valid_pixels = r > 0
    plant_mask = valid_pixels & (index > threshold)
    soil_mask = valid_pixels & (index <= threshold)

    def calculate_means(mask):
        if not np.any(mask): 
            return [np.nan] * (4 if is_satellite else 3)
        if is_satellite:
            return [r[mask].mean(), g[mask].mean(), b[mask].mean(), nir[mask].mean()]
        return [r[mask].mean(), g[mask].mean(), b[mask].mean()]

    plant_colors = calculate_means(plant_mask)
    soil_colors = calculate_means(soil_mask)

    cols = ['Red', 'Green', 'Blue', 'NIR'] if is_satellite else ['Red', 'Green', 'Blue']
    data = {'Imagename': img_name}
    for i, col in enumerate(cols):
        data[f'Plant_{col}_Mean'] = plant_colors[i]
        data[f'Soil_{col}_Mean'] = soil_colors[i]
    
    return pd.Series(data)

# --- DIRECTORY WALKER ---

def run_manual_structure_analysis(base_path):
    all_results = []
    # Targeted research locations for your corn yield project
    RESEARCH_SITES = {'Scottsbluff', 'Ames', 'Crawfordville', 'Lincoln'} 
    
    if not os.path.exists(base_path):
        print(f"Error: Path '{os.path.abspath(base_path)}' not found.")
        return

    for source_type in ['UAV', 'Satellite']:
        source_path = os.path.join(base_path, source_type)
        if not os.path.exists(source_path): continue

        print(f"Analyzing {source_type} data...")

        for root, dirs, files in os.walk(source_path):
            for file in files:
                file_path = os.path.join(root, file)
                ext = file.lower()
                
                try:
                    res = None
                    if source_type == 'Satellite' and ext.endswith(('.tif', '.tiff')):
                        res = process_satellite(file_path)
                    elif source_type == 'UAV' and ext.endswith(('.png', '.jpg', '.jpeg')):
                        res = process_uav(file_path)
                    
                    if res is not None:
                        path_parts = root.split(os.sep)
                        res['Source'] = source_type
                        res['TPS'] = path_parts[-1] if len(path_parts) > 0 else "N/A"
                        
                        # Identify location based on folder name or research site list
                        detected_loc = path_parts[-2] if len(path_parts) > 1 else "Unknown"
                        res['Location'] = detected_loc if detected_loc in RESEARCH_SITES else f"Other ({detected_loc})"
                        
                        all_results.append(res)
                except Exception as e:
                    print(f"Error in {file}: {e}")

    if all_results:
        pd.DataFrame(all_results).to_csv('manual_structure_results.csv', index=False)
        print(f"Done! Results saved for {len(all_results)} images.")

# --- FILE HANDLERS (Keep original logic) ---
def process_satellite(p):
    with rasterio.open(p) as s:
        b = s.read()
        return get_segmented_stats(os.path.basename(p), b[0], b[1], b[2], b[3], True)

def process_uav(p):
    img = cv.imread(p)
    if img is None: return None
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    return get_segmented_stats(os.path.basename(p), img[:,:,0], img[:,:,1], img[:,:,2], is_satellite=False)

if __name__ == "__main__":
    run_manual_structure_analysis('./')